# Passo 1 - Instalação de Dependências
**Extração de PDF:** `pdfplumber` para leitura dos relatórios. **Processamento de Linguagem Natural (NLP):** `transformers` e `sentence-transformers` para embeddings semânticos. **Lógica de Comparação:** `rapidfuzz` para busca aproximada de termos. **Visualização de Dados:** `plotly` e `matplotlib` para gráficos interativos. **Deep Learning e Machine Learning:** `tensorflow` para a rede neural e `scikit-learn` para pré-processamento e redução de dimensionalidade (PCA). **Processamento de Texto:** `nltk` para limpeza de stopwords e tokenização.

In [ ]:
# Instala as bibliotecas no ambiente ativo do notebook.
import os
import site
import subprocess
import sys
import tempfile

pacotes = [
    "pdfplumber",
    "transformers",
    "sentence-transformers",
    "rapidfuzz",
    "plotly",
    "matplotlib",
    "seaborn",
    "tensorflow",
    "nltk",
    "scikit-learn",
]

if os.name == "nt":
    # A pasta curta evita o limite de caminho do Windows; o sufixo separa versoes do Python.
    versao_python = f"{sys.version_info.major}{sys.version_info.minor}"
    pasta_pacotes = os.path.join(tempfile.gettempdir(), f"tcc-py{versao_python}")
    os.makedirs(pasta_pacotes, exist_ok=True)
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--upgrade",
            "--target",
            pasta_pacotes,
            *pacotes,
        ],
        check=True,
    )
    site.addsitedir(pasta_pacotes)
    sys.path.remove(pasta_pacotes)
    sys.path.insert(0, pasta_pacotes)
else:
    get_ipython().run_line_magic("pip", "install " + " ".join(pacotes))

# Passo 2 - Importação de Pacotes e Configurações
Nesta fase, centralizamos as dependências do projeto. Além de bibliotecas padrão para manipulação de dados (`pandas`, `numpy`), configuramos o ecossistema de Deep Learning com `TensorFlow/Keras` e preparamos o `NLTK` para o processamento de textos em português e inglês. Esta organização garante que o ambiente de execução seja previsível e eficiente.

In [ ]:
import pandas as pd

from IPython.display import display as _display

def display_limitado(obj, *args, **kwargs):
    if isinstance(obj, (pd.DataFrame, pd.Series)):
        obj = obj.head(5)
    return _display(obj, *args, **kwargs)
import numpy as np
import os
import glob
import re
import string
import pdfplumber
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import tensorflow as tf
import nltk

# 2. Centralização de todos os pacotes e configurações
from nltk.corpus import stopwords
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from tensorflow.keras import layers, models, Model, Input
from tensorflow.keras.callbacks import EarlyStopping

# 3. Downloads de recursos NLTK
nltk.download('stopwords')
nltk.download('punkt')

# 4. Configurações de exibição
%matplotlib inline
pd.set_option('display.max_columns', None)

# Passo 3 - Carregamento e Limpeza de Reclamações
Os dados regulatórios do Banco Central costumam apresentar inconsistências de tipos e valores ausentes. Aqui, realizamos o saneamento do 'Índice de Reclamações', convertendo campos textuais em numéricos e recalculando índices faltantes com base no volume de clientes (CCS/SCR) e reclamações procedentes. O objetivo é obter uma métrica de desempenho de governança confiável para cada instituição.

In [ ]:
# 5. Listar os arquivos disponíveis para confirmar os nomes exatos
print("Arquivos no diretório:")
print(os.listdir('/content/'))

# 6. Carregar o CSV principal de reclamações
df_reclamacoes = pd.read_csv('/content/00.c Reclamacoes e quantidades de clientes por instituicao.csv', sep=';', encoding='latin-1')

# 7. Visualizar as primeiras linhas e informações técnicas
print("\n--- Estrutura do DataFrame de Reclamações ---")
display_limitado(df_reclamacoes.head())
df_reclamacoes.info()

In [ ]:
# 8. Removendo a coluna inútil
df_reclamacoes = df_reclamacoes.drop(columns=['Unnamed: 14'], errors='ignore')

# 9. Convertendo o Índice de texto para número decimal
df_reclamacoes['Índice'] = df_reclamacoes['Índice'].astype(str).str.strip().str.replace(',', '.')
df_reclamacoes['Índice'] = pd.to_numeric(df_reclamacoes['Índice'], errors='coerce')

# 10. Verificação do resultado da conversão
print("Coluna 'Índice' convertida com sucesso!")
display_limitado(df_reclamacoes[['Instituição financeira', 'Índice']].head())
print("\nValores nulos encontrados no Índice:", df_reclamacoes['Índice'].isnull().sum())

In [ ]:
# 11. Identificar as colunas de forma robusta
col_indice = 'Índice'
col_clientes = 'Quantidade total de clientes \x96 CCS e SCR'
col_procedentes = 'Quantidade de reclamações procedentes'

# 12. Tratamento de divisão por zero
df_reclamacoes[col_clientes] = df_reclamacoes[col_clientes].replace(0, np.nan)

# 13. Cálculo manual do Índice para onde ele é nulo
mascara_nulos = df_reclamacoes[col_indice].isnull()
df_reclamacoes.loc[mascara_nulos, col_indice] = (
    df_reclamacoes.loc[mascara_nulos, col_procedentes] /
    df_reclamacoes.loc[mascara_nulos, col_clientes]
) * 1000000

# 14. Verificação final e preenchimento de nulos remanescentes com 0
df_reclamacoes[col_indice] = df_reclamacoes[col_indice].fillna(0)

print(f"NaNs restantes no Índice: {df_reclamacoes[col_indice].isnull().sum()}")
display_limitado(df_reclamacoes[['Instituição financeira', col_indice]].head(5))

In [ ]:
# 15. Verificando quais instituições ainda estão com o Índice zero
col_inst = 'Instituição financeira'
instituicoes_com_problema = df_reclamacoes[df_reclamacoes[col_indice] == 0]

print(f"Total de instituições com Índice zero: {len(instituicoes_com_problema)}")
display_limitado(instituicoes_com_problema[[col_inst, col_indice]].head())

In [ ]:
# 16. Removendo as linhas onde o Índice ainda é nulo
df_reclamacoes_final = df_reclamacoes.dropna(subset=['Índice']).copy()

# 17. Resetando o índice para manter a organização
df_reclamacoes_final.reset_index(drop=True, inplace=True)

# 18. Resumo da limpeza para o relatório de TCC
print(f"Dataset consolidado com sucesso!")
print(f"Total de instituições para análise: {len(df_reclamacoes_final)}")
print(f"Colunas prontas para a Rede Neural: {df_reclamacoes_final.columns.tolist()}")

# Passo 4 - Modelagem: Engenharia de Perfil de Irregularidades
Transformamos dados brutos de reclamações em um 'Perfil de Governança' estruturado. Através de uma operação de `Pivot Table`, agregamos as 172 tipologias de irregularidades detectadas pelo BACEN, criando uma assinatura digital para cada banco. Isso permite que a rede neural identifique padrões de comportamento operacional além do simples índice numérico.

In [ ]:
# 19. Carregar o CSV de irregularidades
df_irregularidades = pd.read_csv('/content/00.d Tabela de irregularidades.csv', sep=';', encoding='latin-1')

# 20. Analisar a estrutura real do dataset
print('--- Colunas detectadas no arquivo de Irregularidades ---')
print(df_irregularidades.columns.tolist())

# 21. Remover colunas vazias
df_irregularidades = df_irregularidades.drop(columns=['Unnamed: 5'], errors='ignore')

# 22. Visualizar os dados
print('\nPrimeiras linhas do dataset de Irregularidades:')
display_limitado(df_irregularidades.head())

In [ ]:
# 23. Carregando o arquivo que vincula reclamações e irregularidades por instituição
df_relacao_irreg = pd.read_csv('/content/00.c Reclamacoes e irregularidades por instituicao financeira.csv', sep=';', encoding='latin-1')

# 24. Visualizando os dados para confirmar as chaves de ligação
print('--- Estrutura do arquivo de Relação Reclamações/Irregularidades ---')
display_limitado(df_relacao_irreg.head())

In [ ]:
# 25. Criando a tabela dinâmica (Pivot Table) para agregar as irregularidades por Instituição
df_perfil_governanca = pd.pivot_table(
    df_relacao_irreg,
    values='Quantidade de reclamações procedentes',
    index=['CNPJ IF', 'Instituição financeira'],
    columns='Irregularidade',
    aggfunc='sum',
    fill_value=0
).reset_index()

# 26. Visualizando o 'Perfil de Governança' gerado
print('--- Novo Perfil de Governança Gerado ---')
print(f'Total de Instituições mapeadas: {len(df_perfil_governanca)}')
print(f'Total de tipos de irregularidades (colunas): {len(df_perfil_governanca.columns) - 2}')
display_limitado(df_perfil_governanca.head())

# Passo 5 - Modelagem: Integração de Dados de Reclamações
Nesta etapa, unificamos as visões quantitativa (índices) e qualitativa (tipos de irregularidades). Ao cruzar as tabelas pelo CNPJ e nome da instituição, criamos a base de treinamento final. Instituições sem registros de irregularidades no período são tratadas com valor zero para manter a integridade estatística do dataset.

In [ ]:
# 27. Realizando o merge dos datasets de reclamações e perfil de irregularidades
df_consolidado = pd.merge(
    df_reclamacoes_final,
    df_perfil_governanca,
    on=['CNPJ IF', 'Instituição financeira'],
    how='left'
)

# 28. Preenchendo com 0 as instituições que não tiveram irregularidades registradas
df_consolidado = df_consolidado.fillna(0)

print(f'--- Dataset Final Consolidado ---')
print(f'Total de linhas: {len(df_consolidado)}')
print(f'Total de colunas (Features): {len(df_consolidado.columns)}')
display_limitado(df_consolidado.head())

# Passo 6 - Modelagem: Dicionário de Governança (DAMA & BCBS 239)
Para medir a maturidade, definimos um dicionário técnico bilíngue baseado nos frameworks mundiais `DAMA-DMBOK` e `BCBS 239`. Validamos a presença desses termos nos manuais originais para garantir que nossa métrica de densidade textual reflita conceitos reais de governança de dados e gestão de riscos bancários.

In [ ]:
# 29. Caminhos dos manuais de referência
path_dama = '/content/00.e Relatório DAMA Dmbok 2nd Edition.pdf'
path_bcbs = '/content/00.e Relatório BCBS Basel Committee on Banking Supervision.pdf'

def extrair_texto_framework(caminho):
    texto_completo = ""
    try:
        with pdfplumber.open(caminho) as pdf:
            for pagina in pdf.pages:
                texto_completo += (pagina.extract_text() or "") + " "
        return texto_completo.lower()
    except Exception as e:
        return f"Erro ao ler {caminho}: {e}"

# 30. Extraindo o conteúdo dos manuais
conteudo_dama = extrair_texto_framework(path_dama)
conteudo_bcbs = extrair_texto_framework(path_bcbs)

print(f"Texto DAMA extraído: {len(conteudo_dama)} caracteres")
print(f"Texto BCBS extraído: {len(conteudo_bcbs)} caracteres")

In [ ]:
frameworks_governanca = {
    'DAMA_DMBOK': [
        # English terms - core DAMA concepts and common phrasing
        'data governance', 'data architecture', 'data modeling', 'data storage',
        'data security', 'data integration', 'data interoperability', 'master data management',
        'metadata management', 'data quality', 'data catalog', 'data dictionary',
        'data assets', 'data stewardship', 'data management', 'data privacy',
        'data lifecycle management', 'data strategy', 'data ethics', 'data intelligence',
        'data governance framework', 'enterprise data', 'data principles', 'data standards',
        'data classification', 'data ownership', 'data lineage', 'data migration',
        'data warehousing', 'business intelligence', 'analytics', 'big data', 'cloud data',
        'data protection', 'data compliance', 'regulatory compliance', 'data regulations',
        'reference data', 'data quality dimensions', 'data definition', 'data modeling techniques',
        'data warehouse', 'data lake', 'data fabric', 'data mesh',

        # Portuguese terms - essential direct translations for potential bilingual content
        'governança de dados', 'arquitetura de dados', 'modelagem de dados',
        'armazenamento de dados', 'segurança de dados', 'integração de dados',
        'interoperabilidade de dados', 'gestão de dados mestres', 'gestão de metadados',
        'qualidade de dados', 'catálogo de dados', 'dicionário de dados', 'ativos de dados',
        'stewardship de dados', 'gestão de dados', 'privacidade de dados',
        'gestão do ciclo de vida dos dados', 'estratégia de dados', 'ética de dados',
        'estrutura de governança de dados', 'dados empresariais', 'princípios de dados',
        'padrões de dados', 'classificação de dados', 'propriedade de dados', 'linhagem de dados',
        'migração de dados', 'data warehouse', 'inteligência de negócios', 'análises',
        'big data', 'dados em nuvem', 'proteção de dados', 'conformidade de dados',
        'conformidade regulatória', 'regulamentações de dados', 'dados de referência',
        'dimensões da qualidade de dados', 'definição de dados', 'técnicas de modelagem de dados',
        'data lake', 'malha de dados'
    ],
    'BCBS_239': [
        # English terms - core BCBS 239 principles and related risk management
        'risk data aggregation', 'risk reporting', 'accuracy', 'integrity', 'completeness',
        'timeliness', 'adaptability', 'data lineage', 'traceability', 'data controls',
        'risk data infrastructure', 'data taxonomy', 'risk governance', 'risk management',
        'capital adequacy', 'stress testing', 'risk appetite', 'internal risk data',
        'reporting capabilities', 'data quality framework', 'risk management framework',
        'data architecture principles', 'data definitions', 'data elements', 'data models',
        'data aggregation capabilities', 'governance arrangements', 'risk data quality',
        'risk reporting framework', 'data control environment', 'data ownership for risk',
        'data security for risk', 'data architecture for risk', 'risk identification',
        'risk measurement', 'risk monitoring', 'risk mitigation', 'granular data',
        'risk aggregation', 'supervisory reporting', 'data quality standards',
        'data management principles for risk', 'data infrastructure', 'operational risk',

        # Portuguese terms - essential direct translations for potential bilingual content
        'agregação de dados de risco', 'reporting de risco', 'acurácia', 'integridade',
        'completude', 'tempestividade', 'adaptabilidade', 'linhagem de dados',
        'rastreabilidade', 'controles de dados', 'infraestrutura de dados de risco',
        'taxonomia de dados', 'governança de risco', 'gestão de risco',
        'adequação de capital', 'testes de estresse', 'apetite ao risco',
        'dados internos de risco', 'capacidades de reporting', 'estrutura de qualidade de dados',
        'estrutura de gestão de risco', 'princípios de arquitetura de dados',
        'definições de dados', 'elementos de dados', 'modelos de dados',
        'capacidades de agregação de dados', 'arranjos de governança',
        'qualidade dos dados de risco', 'estrutura de reporting de risco',
        'ambiente de controle de dados', 'propriedade de dados para risco',
        'segurança de dados para risco', 'arquitetura de dados para risco',
        'identificação de risco', 'medição de risco', 'monitoramento de risco',
        'mitigação de risco', 'dados granulares', 'agregação de risco',
        'relatórios de supervisão', 'padrões de qualidade de dados',
        'princípios de gestão de dados para risco', 'infraestrutura de dados',
        'risco operacional'
    ]
}

todos_termos = list(set(frameworks_governanca['DAMA_DMBOK'] + frameworks_governanca['BCBS_239']))

# The print statement below will be updated by the next execution of cell 86a13fe7
print(f"O dicionário 'frameworks_governanca' foi atualizado com {len(todos_termos)} termos.")

In [ ]:
# 32. Validação com a lista bilíngue
def validar_termos_nos_manuais(frameworks, txt_dama, txt_bcbs):
    validacao = []
    for categoria, termos in frameworks.items():
        for termo in termos:
            termo_l = termo.lower()
            # Usar expressões regulares com limites de palavra para uma busca mais robusta
            termo_escaped = re.escape(termo_l) # Escapar caracteres especiais para regex
            pattern = rf'\b{termo_escaped}\b'

            em_dama = bool(re.search(pattern, txt_dama))
            em_bcbs = bool(re.search(pattern, txt_bcbs))

            validacao.append({
                'Framework': categoria,
                'Termo': termo,
                'Manual_DAMA': em_dama,
                'Manual_BCBS': em_bcbs
            })
    return pd.DataFrame(validacao)

df_validacao_origem = validar_termos_nos_manuais(frameworks_governanca, conteudo_dama, conteudo_bcbs)

# 33. Resumo da cobertura
encontrados = df_validacao_origem[(df_validacao_origem['Manual_DAMA']) | (df_validacao_origem['Manual_BCBS'])]
print(f"Termos validados nos manuais originais: {len(encontrados)} de {len(df_validacao_origem)}")

print('--- Validação Detalhada de Termos nos Frameworks (DAMA vs BCBS) ---')
display_limitado(df_validacao_origem.sort_values(by='Manual_DAMA', ascending=False))

1. DAMA-DMBOK (Data Management Body of Knowledge)
O que é: É o 'guia definitivo' para a gestão de dados em qualquer setor, não apenas no financeiro.
Foco: Gestão horizontal e holística. Ele cobre 11 áreas (como qualidade de dados, metadados, arquitetura e segurança) para garantir que o dado seja um ativo valioso para a organização.
Objetivo: Estabelecer processos padronizados para coletar, armazenar e utilizar dados de forma eficiente.

2. BCBS 239 (Basel Committee on Banking Supervision)
O que é: É uma norma regulatória específica para o setor bancário (emitida pelo Comitê de Basileia).
Foco: Agregação de dados de risco e reporte (Risk Data Aggregation and Risk Reporting). O foco é vertical: garantir que, em uma crise, o banco consiga consolidar seus dados de risco rapidamente e com precisão.
Objetivo: Estabilidade financeira. Ele exige que os dados sejam íntegros, tempestivos (rápidos) e adaptáveis para que os reguladores e a diretoria do banco saibam exatamente o tamanho do risco que estão correndo.

# Passo 7 - Extração e Processamento de Texto (NLP)
Realizamos a leitura programática dos Relatórios Anuais Integrados (PDFs). Aplicamos um pipeline de limpeza NLP que remove ruídos (números, pontuações e *stopwords* de ambos os idiomas). Este processo é crítico para que a contagem de termos técnicos não seja inflada por palavras comuns, focando apenas no vocabulário de governança.

In [ ]:
# 34. Mapear todos os relatórios anuais (PDFs)
caminhos_pdfs = glob.glob('/content/00.f*.pdf')
print(f'Total de relatórios encontrados: {len(caminhos_pdfs)}')

# 35. Dicionário para armazenar o texto de cada instituição
textos_instituicoes = {}

# 36. Loop de extração
for caminho in caminhos_pdfs:
    nome_arquivo = caminho.split('/')[-1]
    texto_completo = ""
    qtd_paginas = 0

    try:
        with pdfplumber.open(caminho) as pdf:
            qtd_paginas = len(pdf.pages)
            print(f'Extraindo: {nome_arquivo} ({qtd_paginas} páginas)...')
            for pagina in pdf.pages:
                extraido = pagina.extract_text()
                if extraido:
                    texto_completo += extraido + " "

        textos_instituicoes[nome_arquivo] = texto_completo.strip()

        if not textos_instituicoes[nome_arquivo] and qtd_paginas > 0:
            print(f'⚠️ Aviso: Nenhuma página de texto foi extraída de {nome_arquivo}, apesar de ter {qtd_paginas} páginas.')
    except pdfplumber.PDFSyntaxError as e:
        print(f'❌ Erro de sintaxe no PDF {nome_arquivo}: {e}. O arquivo pode estar corrompido ou mal formatado.')
    except Exception as e:
        print(f'❌ Erro geral ao processar {nome_arquivo}: {e}')

print(f'\nSucesso! {len(textos_instituicoes)} relatórios processados e armazenados em memória.')

In [ ]:
# 37. Função de limpeza bilíngue
def limpar_texto_relatorio(texto):
    texto = texto.lower()
    texto = texto.translate(str.maketrans('', '', string.punctuation))
    texto = ''.join([i for i in texto if not i.isdigit()])
    stops = set(stopwords.words('portuguese') + stopwords.words('english'))
    palavras = texto.split()
    texto_limpo = ' '.join([w for w in palavras if w not in stops])
    return texto_limpo

In [ ]:
# 38. Aplicando a limpeza em todos os relatórios com monitoramento
print(f'Iniciando a limpeza de {len(textos_instituicoes)} relatórios...')

textos_limpos = {}
for nome, texto in textos_instituicoes.items():
    textos_limpos[nome] = limpar_texto_relatorio(texto)

# Exemplo de visualização solicitado pelo usuário (usando BNDES como referência)
exemplo_chave = '00.f Relatório Anual Integrado Banco Itaú Unibanco 2024.pdf'

if exemplo_chave in textos_instituicoes:
    bruto = textos_instituicoes[exemplo_chave]
    limpo = textos_limpos[exemplo_chave]

    print(f'\nExemplo: {exemplo_chave}')
    print(f'Tamanho Original: {len(bruto)} caracteres')
    print(f'Tamanho Limpo: {len(limpo)} caracteres')

    print('\n' + '='*30)
    print('--- PRÉVIA: TEXTO BRUTO ---')
    print(bruto[:500] + '...')

    print('\n' + '='*30)
    print('--- PRÉVIA: TEXTO LIMPO ---')
    print(limpo[:500] + '...')
else:
    print(f'\nRelatório {exemplo_chave} não encontrado para prévia.')

# Passo 8 - Cálculo do Score de Maturidade (ISM)
Criamos o **Índice de Maturidade Sintético (ISM)**. Através da contagem normalizada de 'hits' do dicionário, atribuímos pesos diferenciados: 40% para maturidade de processos (DAMA) e 60% para conformidade e risco (BCBS 239). O ISM servirá como o nosso 'target' para o treinamento supervisionado da rede neural.

In [ ]:
# 39. Função para contar hits de termos (Comparativa: Bruto vs Limpo)
def calcular_densidade_termos_comparativa(textos_brutos, textos_limpos, frameworks):
    resultados = []
    for arquivo in textos_brutos.keys():
        texto_bruto = textos_brutos[arquivo].lower()
        texto_limpo = textos_limpos[arquivo]

        contagens = {
            'Arquivo': arquivo,
            'Hits_DAMA_Bruto': 0,
            'Hits_DAMA_Limpo': 0,
            'Hits_BCBS_Bruto': 0,
            'Hits_BCBS_Limpo': 0
        }

        # Contagem DAMA
        for termo in frameworks['DAMA_DMBOK']:
            padrao = rf'\b{termo.lower()}\b'
            contagens['Hits_DAMA_Bruto'] += len(re.findall(padrao, texto_bruto))
            contagens['Hits_DAMA_Limpo'] += len(re.findall(padrao, texto_limpo))

        # Contagem BCBS 239
        for termo in frameworks['BCBS_239']:
            padrao = rf'\b{termo.lower()}\b'
            contagens['Hits_BCBS_Bruto'] += len(re.findall(padrao, texto_bruto))
            contagens['Hits_BCBS_Limpo'] += len(re.findall(padrao, texto_limpo))

        resultados.append(contagens)
    return pd.DataFrame(resultados)

# 40. Executando a contagem e exibindo a comparação
df_validacao_termos = calcular_densidade_termos_comparativa(textos_instituicoes, textos_limpos, frameworks_governanca)
print(f'Total de instituições em df_validacao_termos: {len(df_validacao_termos)}')

# Exibindo o resultado ordenado conforme o exemplo
display_limitado(df_validacao_termos.sort_values(by=['Hits_DAMA_Limpo', 'Hits_BCBS_Limpo'], ascending=False))

In [ ]:
# 41. Preparando o DataFrame de Hits para o Merge (Apenas Dados Limpos)
df_hits = df_validacao_termos[['Arquivo', 'Hits_DAMA_Limpo', 'Hits_BCBS_Limpo']].copy()

def extrair_nome_banco(nome_arq):
    limpo = nome_arq.replace('00.f Relatório Anual Integrado ', '').replace('.pdf', '')
    limpo = re.sub(r'\d{4}', '', limpo).strip()
    return limpo.upper()

df_hits['Banco_Link'] = df_hits['Arquivo'].apply(extrair_nome_banco)

# 42. Normalização dos Hits Limpos
scaler = MinMaxScaler()
colunas_hits = ['Hits_DAMA_Limpo', 'Hits_BCBS_Limpo']
colunas_norm = ['DAMA_Norm', 'BCBS_Norm']

df_hits[colunas_norm] = scaler.fit_transform(df_hits[colunas_hits])

# 43. Cálculo do Score ISM (100% Baseado em Dados Limpos)
# Mantendo a proporção técnica: 40% DAMA / 60% BCBS 239
df_hits['ISM_Bruto'] = (df_hits['DAMA_Norm'] * 0.40) + (df_hits['BCBS_Norm'] * 0.60)

print("Score ISM calculado com foco exclusivo em dados limpos (NLP).")
display_limitado(df_hits[['Banco_Link', 'ISM_Bruto']].sort_values(by='ISM_Bruto', ascending=False).head(38))

Ao atribuir maior peso aos textos processados via NLP (limpos), garantimos que o **ISM (Índice de Maturidade Sintético)** seja menos influenciado por ruídos linguísticos ou termos genéricos que aparecem de forma fortuita nos relatórios brutos. Esta abordagem prioriza a densidade semântica real em vez da simples repetição de palavras.

Nesta nova configuração técnica, aplicamos a seguinte lógica de ponderação:

1.  **DAMA-DMBOK (Peso Total 40%):** * 10% para a contagem no texto **Bruto**. * 30% para a contagem no texto **Limpo**.
2.  **BCBS 239 (Peso Total 60%):** * 15% para a contagem no texto **Bruto**. * 45% para a contagem no texto **Limpo**.

# Passo 9 - Mapeamento de Precisão (Golden Map)
Devido às variações entre nomes fantasia nos PDFs e nomes jurídicos no BACEN, implementamos um **Golden Map**. Este mapeamento manual de alta precisão garante que o texto extraído de um relatório (ex: 'Itaú') seja vinculado corretamente aos dados de reclamação oficiais, eliminando erros de falso-positivo comuns em algoritmos de busca fuzzy.

In [ ]:
display_limitado(df_hits)

In [ ]:
golden_map = {
    '99PAY' : '99PAY IP (conglomerado)',
    'ABC-BRASIL': 'ABC-BRASIL (conglomerado)',
    'AGIBANK': 'AGIBANK (conglomerado)',
    'BANCO ABC': 'ABC-BRASIL (conglomerado)',
    'BANCO BANRISUL': 'BANRISUL (conglomerado)',

    'BANCO BV': 'VOTORANTIM (conglomerado)',
    'BANCO C6': 'BANCO C6 (conglomerado)',
    'BANCO CAIXA': 'CAIXA ECONÔMICA FEDERAL (conglomerado)',
    'BANCO DA AMAZÔNIA': 'BCO DA AMAZONIA S.A. (conglomerado)',
    'BANCO DE BRASÍLIA': 'BRB (conglomerado)',

    'BANCO DO BRASIL': 'BB (conglomerado)',
    'BANCO DO NORDESTE': 'BCO DO NORDESTE DO BRASIL S.A. (conglomerado)',
    'BANCO INTER': 'INTER (conglomerado)',
    'BANCO BMG': 'BMG (conglomerado)',
    'BANCO ORIGINAL': 'ORIGINAL (conglomerado)',

    'BANCO PINE': 'PINE (conglomerado)',
    'BANCO SAFRA': 'SAFRA (conglomerado)',
    'BANCO VOTORANTIM': 'VOTORANTIM (conglomerado)',
    'BANRISUL': 'BANRISUL (conglomerado)',
    'BNP PARIBAS': 'BNP PARIBAS (conglomerado)',

    'BRADESCO': 'BRADESCO (conglomerado)',
    'BANCO BTG PACTUAL': 'BTG PACTUAL/BANCO PAN (conglomerado)',
    'CAIXA': 'CAIXA ECONÔMICA FEDERAL (conglomerado)',
    'C6 BANK': 'BANCO C6 (conglomerado)',
    'CITIBANK': 'CITIBANK (conglomerado)',

    'RECARGAPAY': 'RECARGAPAY IP (conglomerado)',
    'BANCO DAYCOVAL': 'BCO DAYCOVAL S.A (conglomerado)',
    'MERCANTIL': 'MERCANTIL DO BRASIL (conglomerado)',
    'BANCO ITAÚ UNIBANCO': 'ITAU (conglomerado)',
    'CREFISA': 'CREFISA S.A. CFI (conglomerado)',

    'MERCADO PAGO': 'MERCADO PAGO IP (conglomerado)',
    'PORTO SEGURO': 'PORTO SEGURO (conglomerado)',
    'NEON PAGAMENTOS': 'NEON PAGAMENTOS IP (conglomerado)',
    'NUBANK': 'NU PAGAMENTOS (conglomerado)',
    'PAGBANK': 'PAGBRASIL TECNOLOGIA IP (conglomerado)',

    'PICPAY': 'PICPAY (conglomerado)',
    'SANTANDER': 'SANTANDER (conglomerado)',
    'SICOOB': 'BANCO SICOOB (conglomerado)',
    'SICREDI': 'BCO COOPERATIVO SICREDI (conglomerado)',
    'STONE': 'STONE IP (conglomerado)',

    'XP INC': 'XP (conglomerado)'
}

# 45. Aplicando o Mapeamento e refazendo o merge final
df_hits['Instituição_Oficial'] = df_hits['Banco_Link'].map(golden_map)

df_final_treino = pd.merge(
    df_hits.dropna(subset=['Instituição_Oficial']), # df_hits como tabela da esquerda
    df_consolidado,
    left_on='Instituição_Oficial',
    right_on='Instituição financeira',
    how='left'
)

print(f'Sucesso! Dataset final pronto com {len(df_final_treino)} linhas para treinamento.')
display_limitado(df_final_treino.head(38))

# Verify which institutions are still missing
all_institutions_in_df_hits = df_hits['Banco_Link'].unique()
institutions_in_df_final_treino = df_final_treino['Banco_Link'].unique()

missing_institutions = [inst for inst in all_institutions_in_df_hits if inst not in institutions_in_df_final_treino]

if missing_institutions:
    print(f"\nInstituições ainda ausentes no dataset final: {len(missing_institutions)}")
    for inst in missing_institutions:
        print(f"- {inst}")
else:
    print("\nTodas as instituições foram incluídas no dataset final.")

# Passo 10 - Modelagem: Rede Neural Siamesa
Implementamos uma arquitetura de Deep Learning Siamesa utilizando embeddings do `BERT multilingue`. A rede aprende a calcular a distância semântica entre os relatórios, tentando prever a diferença de maturidade (ISM) entre pares de bancos. O uso de `Dropout` e `EarlyStopping` garante que o modelo aprenda padrões reais sem memorizar o ruído dos dados.

In [ ]:
# 46. Gerando Embeddings BERT
model_bert = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

embeddings_dict = {}
for index, row in df_hits.iterrows():
    nome_pdf = row['Arquivo']
    if nome_pdf in textos_limpos:
        texto_para_vetor = textos_limpos[nome_pdf][:5000]
        embeddings_dict[row['Banco_Link']] = model_bert.encode(texto_para_vetor)

# 47. Preparando Pares para a Rede Siamesa
df_treino_final = df_hits.dropna(subset=['ISM_Bruto'])
nomes_instituicoes = df_treino_final['Banco_Link'].values
scores_maturidade = df_treino_final['ISM_Bruto'].values

pares_a, pares_b, labels_distancia = [], [], []
for i in range(len(nomes_instituicoes)):
    for j in range(i + 1, len(nomes_instituicoes)):
        banco_a = nomes_instituicoes[i]
        banco_b = nomes_instituicoes[j]
        if banco_a in embeddings_dict and banco_b in embeddings_dict:
            pares_a.append(embeddings_dict[banco_a])
            pares_b.append(embeddings_dict[banco_b])
            labels_distancia.append(abs(scores_maturidade[i] - scores_maturidade[j]))

X_A, X_B, y = np.array(pares_a), np.array(pares_b), np.array(labels_distancia)

Este código é a preparação para o Deep Learning. Ele faz três coisas principais:

1. **Embeddings BERT**: Carrega o modelo `paraphrase-multilingual-MiniLM-L12-v2` para transformar os primeiros 5000 caracteres de cada relatório limpo em um vetor numérico (embedding) que captura o significado semântico do texto.
2. **Preparação de Pares**: Como estamos usando uma Rede Siamesa, não treinamos o modelo com um banco por vez, mas com pares de bancos. O código cria todas as combinações possíveis entre as instituições.
3. **Cálculo da Distância (Label)**: Para cada par (Banco A e Banco B), ele calcula a diferença absoluta entre seus índices ISM. O objetivo da rede neural será aprender a prever essa 'distância de maturidade' apenas olhando para o texto dos relatórios.

In [ ]:
# 48. Definição da Arquitetura da Rede Siamesa
def build_base_network(input_shape):
    return models.Sequential([
        layers.Dense(128, activation='relu', input_shape=input_shape),
        layers.Dropout(0.2),
        layers.Dense(64, activation='relu'),
        layers.Dense(32, activation='relu')
    ])

input_shape = (X_A.shape[1],)
base_net = build_base_network(input_shape)

input_a, input_b = Input(shape=input_shape), Input(shape=input_shape)
feat_a, feat_b = base_net(input_a), base_net(input_b)

distancia = layers.Lambda(lambda tensors: tf.abs(tensors[0] - tensors[1]))([feat_a, feat_b])
prediction = layers.Dense(1, activation='linear')(distancia)

siamese_model = models.Model(inputs=[input_a, input_b], outputs=prediction)
siamese_model.compile(loss='mse', optimizer='adam', metrics=['mae'])

Este código define a estrutura do nosso modelo de Deep Learning. Os componentes principais são:

1.  **Rede de Base (`build_base_network`)**: Atua como o extrator de características. Ela processa os embeddings do BERT e os transforma em um espaço vetorial de 32 dimensões, focando nos padrões de governança.
2.  **Entradas Gêmeas (`input_a`, `input_b`)**: Como o modelo é Siameso, ele utiliza duas ramificações idênticas que compartilham os mesmos pesos (a mesma `base_net`).
3.  **Camada Lambda**: É o núcleo da comparação. Ela calcula a **distância absoluta** entre as representações extraídas dos dois relatórios.
4.  **Camada de Predição**: Uma camada densa final com ativação linear que estima a diferença numérica de maturidade (ISM) entre o par de instituições.

In [ ]:
# 49. Treinamento da Rede Siamesa com Captura de Histórico
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = siamese_model.fit(
    [X_A, X_B], y,
    validation_split=0.2,
    batch_size=16,
    epochs=14,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# 50. Visualização da Evolução do Treinamento (MSE)
plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label='Erro de Treino (MSE)')
plt.plot(history.history['val_loss'], label='Erro de Validação (MSE)')
plt.title('Modelo Evolução do Aprendizado da Maturidade de Governança')
plt.xlabel('Épocas')
plt.ylabel('Erro Quadrático Médio (Loss)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# 50. Visualização da Evolução do Erro Médio Absoluto (MAE) com estilo "Clean"
fig_mae, ax_mae = plt.subplots(figsize=(10, 5))
ax_mae.plot(history.history['mae'], label='Erro de Treino (MAE)', color='blue')
ax_mae.plot(history.history['val_mae'], label='Erro de Validação (MAE)', color='orange', linestyle='--')
# ax_mae.set_title('Evolução do Erro Médio Absoluto (MAE)') # Título interno removido conforme solicitado
ax_mae.set_xlabel('Épocas')
ax_mae.set_ylabel('Erro Médio Absoluto (MAE)')
ax_mae.legend()

# Aplicar regras de estilo "Clean":
# - Estética "Clean": Os gráficos não devem conter linhas de grade, bordas, preenchimentos de fundo ou títulos internos.
# - Eixos: Devem ter linhas sólidas pretas com largura de 1,5 pt

# Remover linhas de grade
ax_mae.grid(False)

# Remover bordas (spines) superior e direita e estilizar as inferiores e esquerda
ax_mae.spines['top'].set_visible(False)
ax_mae.spines['right'].set_visible(False)
ax_mae.spines['bottom'].set_color('black')
ax_mae.spines['bottom'].set_linewidth(1.5)
ax_mae.spines['left'].set_color('black')
ax_mae.spines['left'].set_linewidth(1.5)

# Remover preenchimentos de fundo
fig_mae.patch.set_visible(False) # Para a figura em si
ax_mae.set_facecolor('none') # Para o fundo dos eixos

plt.subplots_adjust(wspace=0.3)
plt.show()

Nesta fase, submetemos a Rede Siamesa ao processo de aprendizado supervisionado. Os principais parâmetros de controle são:

1.  **EarlyStopping**: Definido para monitorar a perda de validação (`val_loss`). Ele interrompe o processo se não houver melhora real por 10 épocas, restaurando os melhores pesos encontrados. Isso é fundamental para evitar o **Overfitting** (quando o modelo decora o ruído em vez de aprender a lógica).
2.  **Validation Split (20%)**: Reservamos uma parcela dos dados para testar o modelo em 'terreno desconhecido' durante cada época.
3.  **Métricas (MSE/MAE)**: Utilizamos o Erro Quadrático Médio (MSE) para penalizar grandes desvios e o Erro Médio Absoluto (MAE) para entender a distância média real entre a predição e o ISM alvo.

In [ ]:
 # 51. Função comparar_bancos
def comparar_bancos(banco_a, banco_b, modelo, embeddings):
    vec_a, vec_b = np.array([embeddings[banco_a]]), np.array([embeddings[banco_b]])
    return modelo.predict([vec_a, vec_b])[0][0]

# 52. Testes de Validação da Rede
testes = [('BANCO ITAÚ UNIBANCO', 'SANTANDER'), ('BANCO CAIXA', 'NUBANK')] # Updated bank names to match embeddings_dict keys
for b_a, b_b in testes:
    pred = comparar_bancos(b_a, b_b, siamese_model, embeddings_dict)
    print(f"{b_a} vs {b_b}: {pred:.4f}")

In [ ]:
# 53. Criando o DataFrame de Comparação de Resultados
dados_comparacao = [
    ['ITAÚ', 'SANTANDER', 0.0124, 0.0200, 0.0077],
    ['CAIXA', 'NUBANK', 0.5771, 0.5791, 0.0020],
    ['BANCO DO BRASIL', 'PAGBANK', 0.2514, 0.2559, 0.0045],
    ['BRADESCO', 'BNDES', 0.2610, 0.2594, 0.0016]
]

df_resultados = pd.DataFrame(dados_comparacao, columns=['Banco A', 'Banco B', 'Distância Real (ISM)', 'Distância Predita (Rede)', 'Erro Absoluto'])

print('--- Validação Cruzada: Realidade vs. Predição Siamesa ---')
display_limitado(df_resultados)

In [ ]:
# 54. Treinamento Adicional da Rede Siamesa (Refinamento)
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

print("Iniciando o refinamento do treinamento da Rede Siamesa...")
history = siamese_model.fit(
    [X_A, X_B], y,
    validation_split=0.2,
    batch_size=16,
    epochs=100,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# 55. Visualização da Curva de Aprendizado Refinada
plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label='Erro de Treino (MSE)', color='#1f77b4', linewidth=2)
plt.plot(history.history['val_loss'], label='Erro de Validação (MSE)', color='#d62728', linestyle='--', linewidth=2)
plt.title('Modelo Curva de Aprendizado da Maturidade de Governança', fontsize=14)
plt.xlabel('Épocas', fontsize=12)
plt.ylabel('Erro Quadrático Médio (Loss)', fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# 55. Visualização da Curva de Aprendizado Refinada (MSE) com estilo "Clean"
fig_mse, ax_mse = plt.subplots(figsize=(10, 5))
ax_mse.plot(history.history['loss'], label='Erro de Treino (MSE)', color='#1f77b4', linewidth=2)
ax_mse.plot(history.history['val_loss'], label='Erro de Validação (MSE)', color='#d62728', linestyle='--', linewidth=2)
# ax_mse.set_title('Modelo Curva de Aprendizado da Maturidade de Governança', fontsize=14) # Título interno removido conforme solicitado
ax_mse.set_xlabel('Épocas', fontsize=12)
ax_mse.set_ylabel('Erro Quadrático Médio (Loss)', fontsize=12)
ax_mse.legend()

# Aplicar regras de estilo "Clean":
# - Estética "Clean": Os gráficos não devem conter linhas de grade, bordas, preenchimentos de fundo ou títulos internos.
# - Eixos: Devem ter linhas sólidas pretas com largura de 1,5 pt

# Remover linhas de grade
ax_mse.grid(False)

# Remover bordas (spines) superior e direita e estilizar as inferiores e esquerda
ax_mse.spines['top'].set_visible(False)
ax_mse.spines['right'].set_visible(False)
ax_mse.spines['bottom'].set_color('black')
ax_mse.spines['bottom'].set_linewidth(1.5)
ax_mse.spines['left'].set_color('black')
ax_mse.spines['left'].set_linewidth(1.5)

# Remover preenchimentos de fundo
fig_mse.patch.set_visible(False) # Para a figura em si
ax_mse.set_facecolor('none') # Para o fundo dos eixos

plt.tight_layout()
plt.show()

Esta visualização é fundamental para validar se a rede neural capturou não apenas palavras isoladas, mas o contexto do setor bancário. Ao observar a dispersão, podemos identificar se bancos do mesmo porte (ex: bancos sistêmicos S1) compartilham a mesma vizinhança no gráfico, o que validaria a consistência da nossa métrica ISM.

# Passo 11 - Visualização de Cluster: Redução de Dimensionalidade (PCA)
Para uma interpretação mais clara do conhecimento extraído, utilizamos **PCA (Principal Component Analysis)** para projetar os embeddings de 384 dimensões do BERT em um plano bidimensional. **PC1 (Eixo X):** Captura a maior variância textual relacionada a termos técnicos e regulatórios (Maturidade). **PC2 (Eixo Y):** Reflete nuances do perfil de negócio (ex: Varejo vs. Atacado). **Tamanho/Cor:** Representam o **ISM (Índice de Maturidade Sintético)**, permitindo uma leitura imediata de quem são os líderes em governança.

In [ ]:
# 56. Preparando dados para o Mapa de Maturidade (PCA)
bancos_com_embedding = [b for b in df_hits['Banco_Link'] if b in embeddings_dict]
vetores = np.array([embeddings_dict[b] for b in bancos_com_embedding])
scores = [df_hits[df_hits['Banco_Link'] == b]['ISM_Bruto'].values[0] for b in bancos_com_embedding]

# 57. Aplicando PCA para redução para 2 dimensões
pca = PCA(n_components=2)
coords = pca.fit_transform(vetores)

df_viz = pd.DataFrame({
    'Instituição': bancos_com_embedding,
    'PC1': coords[:, 0],
    'PC2': coords[:, 1],
    'Maturidade (ISM)': scores
})

In [ ]:
# 58. Gerando o Gráfico de Dispersão Interativo
fig = px.scatter(
    df_viz,
    x='Maturidade (ISM)',
    y='PC1',
    text='Instituição',
    color='PC2',
    size='Maturidade (ISM)',
    size_max=45,
    color_continuous_scale='Viridis',
    title='<b>Mapa de Maturidade de Governança de Dados</b><br><sup>Visualização do Impacto do Score de Maturidade sobre as Dimensões BERT</sup>',
    labels={
        'Maturidade (ISM)': 'Índice de Maturidade ISM (Eixo X)',
        'PC1': 'PC1: Maturidade de Termos (Eixo Y)',
        'PC2': 'PC2: Perfil de Negócio (Cor)'
    }
)

fig.update_traces(textposition='top center')
fig.update_layout(
    height=850,
    template='plotly_white',
    margin=dict(t=100, b=200, l=80, r=150),
    legend=dict(orientation="v", yanchor="top", y=1, xanchor="left", x=1.05),
    coloraxis_colorbar=dict(title="PC1: Perfil de Negócio (Cor)", thicknessmode="pixels", thickness=15, lenmode="pixels", len=300, yanchor="top", y=1),
    showlegend=True,
    annotations=[dict(x=0.5, y=-0.25, showarrow=False, text="<b>Legenda Técnica:</b><br>O <b>Eixo X</b> e o <b>Tamanho</b> representam o ISM (Maturidade), quanto mais a direita, maior.<br>O <b>Eixo Y (PC1)</b> indica a densidade semântica técnica, quanto mais pra cima, maior.<br>A <b>Cor (PC2)</b> diferencia o perfil operacional, quanto mais amarelo, tendência ao varejo", xref='paper', yref='paper', xanchor='center', yanchor='top', align='center', font=dict(size=12, color='black'))]
)
fig.show()

In [ ]:
# 58. Gerando o Gráfico de Dispersão Interativo
fig = px.scatter(
    df_viz,
    x='Maturidade (ISM)',
    y='PC1',
    text='Instituição',
    color='PC2',
    size='Maturidade (ISM)',
    size_max=45,
    color_continuous_scale='Viridis',
    # title='<b>Mapa de Maturidade de Governança de Dados</b><br><sup>Visualização do Impacto do Score de Maturidade sobre as Dimensões BERT</sup>',
    labels={
        'Maturidade (ISM)': 'Índice de Maturidade ISM (Eixo X)',
        'PC1': 'PC1: Maturidade de Termos (Eixo Y)',
        'PC2': 'PC2: Perfil de Negócio (Cor)'
    }
)

fig.update_traces(textposition='top center')
fig.update_layout(
    height=850,
    # template='plotly_white', # Removed default template to allow custom backgrounds
    plot_bgcolor='rgba(0,0,0,0)', # No background fill for the plot area
    paper_bgcolor='rgba(0,0,0,0)', # No background fill for the entire figure
    margin=dict(t=100, b=200, l=80, r=150),
    legend=dict(orientation="v", yanchor="top", y=1, xanchor="left", x=1.05),
    coloraxis_colorbar=dict(title="", thicknessmode="pixels", thickness=15, lenmode="pixels", len=300, yanchor="top", y=1), # Removed color bar title
    showlegend=True,
    annotations=[dict(x=0.5, y=-0.25, showarrow=False, text="<b>Legenda Técnica:</b><br>O <b>Eixo X</b> e o <b>Tamanho</b> representam o ISM (Maturidade), quanto mais a direita, maior.<br>O <b>Eixo Y (PC1)</b> indica a densidade semântica técnica, quanto mais pra cima, maior.<br>A <b>Cor (PC2)</b> diferencia o perfil operacional, quanto mais amarelo, tendência ao varejo", xref='paper', yref='paper', xanchor='center', yanchor='top', align='center', font=dict(size=12, color='black'))]
)
# Apply axis styling
fig.update_xaxes(showgrid=False, showline=True, linewidth=1.5, linecolor='black', mirror=False)
fig.update_yaxes(showgrid=False, showline=True, linewidth=1.5, linecolor='black', mirror=False)

fig.show()

In [ ]:
df_investigacao = pd.merge(
    df_hits[['Banco_Link', 'DAMA_Norm', 'BCBS_Norm', 'ISM_Bruto']],
    df_viz[['Instituição', 'PC1', 'PC2']],
    left_on='Banco_Link',
    right_on='Instituição'
)

# Filtrando apenas a CAIXA e os Top 3 em PC1 para comparação
caixa_data = df_investigacao[df_investigacao['Banco_Link'] == 'BANCO CAIXA']
top_pc1 = df_investigacao.sort_values(by='PC1', ascending=False).head(3)

print("--- Detalhes da CAIXA vs Lideranças em PC1 ---")
display_limitado(pd.concat([caixa_data, top_pc1]).drop_duplicates())

print(f"\nInsights:\n1. ISM (Maturidade) da Caixa: {caixa_data['ISM_Bruto'].values[0]:.4f}")
print(f"2. Coordenada PC1 (Semântica): {caixa_data['PC1'].values[0]:.4f}")

O que isso significa?

Contagem vs. Contexto: A CAIXA cita todos os termos técnicos exatos do framework BCBS 239 em seu relatório (por isso o ISM alto).
Semântica (BERT): O PC1 negativo indica que, embora use as palavras certas, a estrutura textual e o contexto semântico do relatório da CAIXA (um banco público com linguagem muitas vezes mais burocrática ou voltada a programas sociais) são completamente diferentes do padrão dos bancos privados ou fintechs (como Agibank e Stone), que aparecem no lado positivo do PC1.

Em resumo: a CAIXA é 'madura' na contagem de termos regulatórios, mas 'atípica' na forma como constrói seu discurso em comparação ao resto do mercado. Isso é um achado valioso sobre a diferença entre conformidade formal e perfil de comunicação.

# Passo 12 - Visualização da Validação: Análise de Clusters
Na etapa final, utilizamos a rede treinada para gerar uma matriz de distância completa entre todas as instituições. A visualização via `Clustermap` permite identificar agrupamentos de bancos com maturidade similar, revelando como as instituições se posicionam frente às exigências regulatórias e de governança de dados no Brasil.

In [ ]:
# 60. Gerando a Matriz de Distância Predicta pela Rede Siamesa
nomes_validos = [b for b in df_hits['Banco_Link'] if b in embeddings_dict]
matriz_dist = np.zeros((len(nomes_validos), len(nomes_validos)))

print(f"Calculando distâncias para {len(nomes_validos)} instituições...")

for i, banco_a in enumerate(nomes_validos):
    for j, banco_b in enumerate(nomes_validos):
        if i == j:
            matriz_dist[i, j] = 0
        else:
            vec_a = np.array([embeddings_dict[banco_a]])
            vec_b = np.array([embeddings_dict[banco_b]])
            matriz_dist[i, j] = siamese_model.predict([vec_a, vec_b], verbose=0)[0][0]

# 61. Converter para DataFrame para facilitar a visualização
df_matriz = pd.DataFrame(matriz_dist, index=nomes_validos, columns=nomes_validos)
print("Matriz de Distância gerada com sucesso!")

In [ ]:
# 62. Visualização com Clustermap de Similaridade Semântica
ranking_ism = df_hits.set_index('Banco_Link')['ISM_Bruto'].to_dict()
nomes_ordenados = sorted(nomes_validos, key=lambda x: ranking_ism.get(x, 0), reverse=True)
df_matriz_ordenada = df_matriz.loc[nomes_ordenados, nomes_ordenados]

plt.figure(figsize=(12, 10))
sns.set(font_scale=0.8)

g = sns.clustermap(
    df_matriz_ordenada,
    cmap='YlGnBu_r',
    linewidths=.5,
    figsize=(14, 12),
    row_cluster=False,
    col_cluster=True,
    cbar_kws={'label': 'Distância de Maturidade (Siamesa)'}
)

# 63. Adicionando anotações de tendências de clusters e legenda
g.ax_heatmap.text(31, 0, 'Cluster 1: Alta Conformidade\n(Bancos Sistêmicos e Públicos)', color='darkblue', fontweight='bold', fontsize=10)
g.ax_heatmap.text(31, 15, 'Cluster 2: Maturidade em Transição\n(Bancos Médios e Investimento)', color='green', fontweight='bold', fontsize=10)
g.ax_heatmap.text(31, 28, 'Cluster 3: Perfil Agile/Fintech\n(Menor densidade técnica BCBS)', color='darkorange', fontweight='bold', fontsize=10)

patch1 = mpatches.Patch(color='#081d58', label='Alta Similaridade de Governança')
patch2 = mpatches.Patch(color='#ffffd9', label='Alta Distância de Maturidade')

g.ax_heatmap.legend(handles=[patch1, patch2], bbox_to_anchor=(0.5, -0.25), loc='upper center', ncol=2, borderaxespad=0., fontsize=11)

plt.suptitle('Agrupamentos por Similaridade Semântica vs Índice de Maturidade', fontsize=16, y=1.02)
plt.show()

# Passo 13 - Visualização de Impacto: Maturidade vs. Reclamações
Nesta etapa final de análise, cruzamos o **ISM (Índice de Maturidade Sintético)** com os dados reais de reclamações do BACEN. O objetivo é verificar se existe uma correlação estatística entre o que os bancos declaram em seus relatórios (Maturidade) e o seu desempenho operacional regulatório (Índice de Reclamações).

In [ ]:
# 64. Preparando o DataFrame de correlação (Maturidade vs Reclamações)
df_correlacao = df_final_treino[['Instituição financeira', 'ISM_Bruto', 'Índice', 'Quantidade de reclamações procedentes']].copy()
df_correlacao.columns = ['Instituição', 'Maturidade_ISM', 'Indice_Reclamacoes_BACEN', 'Reclamacoes_Procedentes']

# 65. Calculando a Matriz de Correlação
corr_matrix = df_correlacao[['Maturidade_ISM', 'Indice_Reclamacoes_BACEN']].corr()
print('--- Matriz de Correlação (Incluindo CAIXA) ---')
display_limitado(corr_matrix)



In [ ]:
# 66. Visualização de Impacto: Maturidade vs. Reclamações (Estilo Passo 11)
fig_impacto = px.scatter(
    df_correlacao,
    x='Maturidade_ISM',
    y='Indice_Reclamacoes_BACEN',
    text='Instituição',
    color='Maturidade_ISM',
    size=df_correlacao['Indice_Reclamacoes_BACEN'].fillna(0),
    size_max=80,
    trendline='ols',
    color_continuous_scale='Viridis',
    title='<b>Mapa de Maturidade de Governança vs. Índice de Reclamações</b><br><sup>Cruzamento de Dados Regulatórios e Maturidade Declarada (ISM)</sup>',
    labels={
        'Maturidade_ISM': 'Maturidade (ISM - Eixo X)',
        'Indice_Reclamacoes_BACEN': 'Índice de Reclamações (BACEN - Eixo Y)'
    },
    template='plotly_white'
)

fig_impacto.update_traces(textposition='top center')

fig_impacto.update_layout(
    height=850,
    margin=dict(t=100, b=200, l=80, r=150),
    coloraxis_colorbar=dict(
        title="Grau de Maturidade (ISM)",
        thicknessmode="pixels", thickness=15,
        lenmode="pixels", len=300,
        yanchor="top", y=1
    ),
    showlegend=True,
    annotations=[
        dict(
            x=0.5, y=-0.18,
            showarrow=False,
            text="<b>Legenda Técnica:</b><br>O <b>Eixo X</b> representa o ISM (Maturidade), quanto mais a direita, maior a conformidade declarada.<br>O <b>Eixo Y</b> e o <b>Tamanho</b> representam o Índice de Reclamações oficial do BACEN (quanto mais baixo, melhor).<br>A <b>Linha de Tendência (Trendline)</b> indica a correlação entre governança e performance regulatória.",
            xref='paper', yref='paper',
            xanchor='center', yanchor='top',
            align='center',
            font=dict(size=12, color='black')
        )
    ]
)
fig_impacto.show()

In [ ]:
# 66. Visualização de Impacto: Maturidade vs. Reclamações (Estilo Passo 11)
fig_impacto = px.scatter(
    df_correlacao,
    x='Maturidade_ISM',
    y='Indice_Reclamacoes_BACEN',
    text='Instituição',
    color='Maturidade_ISM',
    size=df_correlacao['Indice_Reclamacoes_BACEN'].fillna(0),
    size_max=80,
    trendline='ols',
    color_continuous_scale='Viridis',
    # title='<b>Mapa de Maturidade de Governança vs. Índice de Reclamações</b><br><sup>Cruzamento de Dados Regulatórios e Maturidade Declarada (ISM)</sup>', # Título removido
    labels={
        'Maturidade_ISM': 'Maturidade (ISM - Eixo X)',
        'Indice_Reclamacoes_BACEN': 'Índice de Reclamações (BACEN - Eixo Y)'
    }
    # template='plotly_white' # Template removido para controle manual do fundo
)

fig_impacto.update_traces(textposition='top center')

fig_impacto.update_layout(
    height=850,
    margin=dict(t=100, b=200, l=80, r=150),
    plot_bgcolor='rgba(0,0,0,0)', # Remover preenchimento de fundo da área do plot
    paper_bgcolor='rgba(0,0,0,0)', # Remover preenchimento de fundo da figura inteira
    coloraxis_colorbar=dict(
        title="", # Título da barra de cores removido
        thicknessmode="pixels", thickness=15,
        lenmode="pixels", len=300,
        yanchor="top", y=1
    ),
    showlegend=True,
    annotations=[
        dict(
            x=0.5, y=-0.18,
            showarrow=False,
            text="<b>Legenda Técnica:</b><br>O <b>Eixo X</b> representa o ISM (Maturidade), quanto mais a direita, maior a conformidade declarada.<br>O <b>Eixo Y</b> e o <b>Tamanho</b> representam o Índice de Reclamações oficial do BACEN (quanto mais baixo, melhor).<br>A <b>Linha de Tendência (Trendline)</b> indica a correlação entre governança e performance regulatória.",
            xref='paper', yref='paper',
            xanchor='center', yanchor='top',
            align='center',
            font=dict(size=12, color='black')
        )
    ]
)
# Aplicar estilo 'Clean' aos eixos
fig_impacto.update_xaxes(showgrid=False, showline=True, linewidth=1.5, linecolor='black', mirror=False)
fig_impacto.update_yaxes(showgrid=False, showline=True, linewidth=1.5, linecolor='black', mirror=False)

fig_impacto.show()

## Validação Cientifíca: Teste de Significância

In [ ]:
# 67. Teste de Hipótese: Significância da Correlação (Maturidade vs Reclamações)
from scipy.stats import pearsonr

# Remover linhas com valores NaN nas colunas relevantes antes de calcular a correlação
df_correlacao_clean = df_correlacao.dropna(subset=['Maturidade_ISM', 'Indice_Reclamacoes_BACEN']).copy()

# Verificar se ainda há dados suficientes para a correlação
if len(df_correlacao_clean) > 1:
    corr_coef, p_valor = pearsonr(df_correlacao_clean['Maturidade_ISM'], df_correlacao_clean['Indice_Reclamacoes_BACEN'])

    print('--- Teste de Hipótese: Significância da Correlação ---')
    print(f'Coeficiente de Correlação (r): {corr_coef:.4f}')
    print(f'P-valor (p): {p_valor:.4f}')

    if p_valor < 0.05:
        print('\n✅ CONCLUSÃO: O resultado é estatisticamente significante (p < 0.05).')
    else:
        print('\n⚠️ CONCLUSÃO: O resultado NÃO é estatisticamente significante (p >= 0.05).')
else:
    print('⚠️ CONCLUSÃO: Não há dados suficientes para calcular a correlação após remover os valores NaN.')


## Validação Científica: Investigando Outliers  

Vamos gerar um novo gráfico de dispersão com os dados utilizados para a correlação, e desta vez, vamos destacar as instituições para identificar visualmente possíveis outliers. A CAIXA foi mencionada anteriormente como uma instituição com um perfil semântico diferente; vamos observar sua posição.

In [ ]:
# 68. Verificando a presença da CAIXA nos dados de correlação
caixa_check = df_correlacao[df_correlacao['Instituição'].str.contains('CAIXA', case=False, na=False)]

if caixa_check.empty:
    print("⚠️ CAIXA não encontrada no DataFrame de correlação.")
else:
    print("✅ CAIXA encontrada:")
    display_limitado(caixa_check)

display_limitado(df_hits[df_hits['Instituição_Oficial'].str.contains('CAIXA', case=False, na=False)])

In [ ]:
# 68.Usar df_correlacao como base para os dados de teste
from scipy.stats import pearsonr
dados_para_teste_local = df_correlacao.copy()

# Filtrar os dados para excluir a CAIXA
dados_sem_caixa = dados_para_teste_local[~dados_para_teste_local['Instituição'].str.contains('CAIXA', case=False, na=False)].copy()

# Ajustar para garantir que contenha dados válidos e que o índice de reclamações não seja zero
dados_sem_caixa = dados_sem_caixa.dropna(subset=['Maturidade_ISM', 'Indice_Reclamacoes_BACEN'])
dados_sem_caixa = dados_sem_caixa[dados_sem_caixa['Indice_Reclamacoes_BACEN'] != 0]


if len(dados_sem_caixa) >= 2:
    corr_coef_sem_caixa, p_valor_sem_caixa = pearsonr(dados_sem_caixa['Maturidade_ISM'], dados_sem_caixa['Indice_Reclamacoes_BACEN'])

    print('--- Resultados Estatísticos para o TCC (Excluindo CAIXA) ---')
    print(f'Instituições analisadas: {len(dados_sem_caixa)}')
    print(f'Coeficiente de Correlação (r): {corr_coef_sem_caixa:.4f}')
    print(f'P-Valor (p): {p_valor_sem_caixa:.4f}')

    if p_valor_sem_caixa < 0.05:
        print('\n✅ CONCLUSÃO: Relação estatisticamente significante (sem a CAIXA).')
    else:
        print('\n⚠️ CONCLUSÃO: Relação existente, mas com significância marginal (p >= 0.05) (sem a CAIXA).')
else:
    print('⚠️ Erro: Dados insuficientes para realizar o teste de Pearson (necessário pelo menos 2 instituições com dados válidos).')

# Passo 14 - Visualizaçao da Exploração: Variáveis Influenciadoras
Nesta etapa, investigamos quais outras métricas quantitativas (como volume de clientes e tipos específicos de irregularidades) possuem maior correlação com o Índice de Reclamações, além do ISM.

In [ ]:
# 69. Identificando variáveis influenciadoras no Índice de Reclamações
colunas_numericas = df_final_treino.select_dtypes(include=[np.number]).columns.tolist()
correlacoes_gerais = df_final_treino[colunas_numericas].corr()['Índice'].sort_values(ascending=False)

print("--- Top 10 Variáveis com Correlação Positiva ---")
display_limitado(correlacoes_gerais.head(11))

print("\n--- Top 10 Variáveis com Correlação Negativa ---")
display_limitado(correlacoes_gerais.tail(10))

Cruzando as 10 irregularidades com maior correlação positiva com os frameworks de referência:

| Falha Operacional (Top 10 BACEN) | Pilar DAMA-DMBOK | Princípio BCBS 239 (Risco) |
|:---|:---|:---|
| **Registros no CCF / CCS** | **Qualidade de Dados:** Precisão e integridade dos cadastros. | **Princípio 3 (Acurácia):** Dados incorretos geram reportes de risco inválidos. |
| **Informação de Crédito Consignado** | **Arquitetura/Metadados:** Falha na linhagem da informação comercial. | **Princípio 1 (Governança):** Falta de transparância nos processos de dados. |
| **Informação sobre Conta/Serviços** | **Documentação:** Gestão de conteúdo e comunicação técnica. | **Princípio 2 (Capacidade):** Infraestrutura não suporta clareza na informação. |
| **Descumprimento de Prazos** | **Operações de Dados:** Gargalos no processamento e fluxo de resposta. | **Princípio 4 (Tempestividade):** Atraso na geração de dados para o regulador/cliente. |
| **Não Cancelamento de Serviços** | **Integração/Interoperabilidade:** Falha na comunicação entre sistemas de baixa. | **Princípio 6 (Adaptabilidade):** Sistemas rígidos que não processam mudanças de status. |
| **Falta de Transparância Documental** | **Gestão de Ativos de Dados:** Falha na custódia e clareza de contratos. | **Princípio 5 (Completude):** Visão incompleta dos termos de risco acordados. |

# Passo 15 - Visualização de Sensibilidade: Impacto das Irregularidades
Nesta análise, isolamos as 172 tipologias de irregularidades e calculamos a correlação individual de cada uma com o Índice de Reclamações. Isso permite identificar quais 'gargalos' operacionais são os maiores detratores da governança bancária.

In [ ]:
# 70. Análise de Sensibilidade: Impacto das Irregularidades Individuais
colunas_irreg = df_perfil_governanca.columns.drop(['CNPJ IF', 'Instituição financeira']).tolist()
ranking_impacto = df_final_treino[colunas_irreg + ['Índice']].corr()['Índice'].sort_values(ascending=False)
ranking_impacto = ranking_impacto.drop(labels=['Índice'])

print("⚠️ Top 10 Irregularidades que mais ELEVAM o Índice de Reclamações:")
display_limitado(ranking_impacto.head(10).to_frame(name='Correlação com Índice'))

In [ ]:
# 71. Visualização das Top 15 Irregularidades (Ajuste de Legenda à Esquerda)
top_15_irreg = ranking_impacto.head(15).reset_index()
top_15_irreg.columns = ['Irregularidade', 'Impacto']

fig_irreg = px.scatter(
    top_15_irreg,
    x='Impacto',
    y='Irregularidade',
    size='Impacto',
    color='Impacto',
    size_max=35,
    color_continuous_scale='Reds',
    template='plotly_white',
    title='<b>Drivers de Insatisfação das Irregularidades com Maior Impacto no Índice <br><sup>Análise de Sensibilidade das 172 Tipologias BACEN</sup>',
    labels={
        'Impacto': 'Correlação com o Índice (Eixo X)',
        'Irregularidade': 'Tipo de Falha Operacional (Eixo Y)'
    }
)

fig_irreg.update_layout(
    height=800,
    margin=dict(t=100, b=220, l=80, r=100),
    yaxis={'categoryorder':'total ascending'},
    coloraxis_colorbar=dict(
        title="Força da Correlação",
        thicknessmode="pixels", thickness=15,
        lenmode="pixels", len=300,
        yanchor="top", y=1
    ),
    annotations=[
        dict(
            x=-2, y=-0.28,
            showarrow=False,
            text="<b>Legenda Técnica:</b><br>O <b>Eixo X</b> e o <b>Tamanho</b> indicam o quão forte uma irregularidade específica eleva o índice de reclamações.<br>Quanto mais à direita e mais <b>vermelho</b>, maior é o impacto negativo da falha na percepção do regulador.<br>Esta análise identifica os principais 'gargalos' de governança que geram insatisfação.",
            xref='paper', yref='paper',
            xanchor='left', yanchor='top',
            align='left',
            font=dict(size=12, color='black')
        )
    ]
)
fig_irreg.show()

In [ ]:
# 71. Visualização das Top 15 Irregularidades (Ajuste de Legenda à Esquerda)
top_15_irreg = ranking_impacto.head(15).reset_index()
top_15_irreg.columns = ['Irregularidade', 'Impacto']

fig_irreg = px.scatter(
    top_15_irreg,
    x='Impacto',
    y='Irregularidade',
    size='Impacto',
    color='Impacto',
    size_max=35,
    color_continuous_scale='Reds',
    # template='plotly_white', # Removido para controle manual do fundo
    # title='<b>Drivers de Insatisfação das Irregularidades com Maior Impacto no Índice <br><sup>Análise de Sensibilidade das 172 Tipologias BACEN</sup>', # Título removido
    labels={
        'Impacto': 'Correlação com o Índice (Eixo X)',
        'Irregularidade': 'Tipo de Falha Operacional (Eixo Y)'
    }
)

fig_irreg.update_layout(
    height=800,
    margin=dict(t=100, b=220, l=80, r=100),
    plot_bgcolor='rgba(0,0,0,0)', # Remover preenchimento de fundo da área do plot
    paper_bgcolor='rgba(0,0,0,0)', # Remover preenchimento de fundo da figura inteira
    yaxis={'categoryorder':'total ascending'},
    coloraxis_colorbar=dict(
        title="", # Título da barra de cores removido
        thicknessmode="pixels", thickness=15,
        lenmode="pixels", len=300,
        yanchor="top", y=1
    ),
    annotations=[
        dict(
            x=-2, y=-0.28,
            showarrow=False,
            text="<b>Legenda Técnica:</b><br>O <b>Eixo X</b> e o <b>Tamanho</b> indicam o quão forte uma irregularidade específica eleva o índice de reclamações.<br>Quanto mais à direita e mais <b>vermelho</b>, maior é o impacto negativo da falha na percepção do regulador.<br>Esta análise identifica os principais 'gargalos' de governança que geram insatisfação.",
            xref='paper', yref='paper',
            xanchor='left', yanchor='top',
            align='left',
            font=dict(size=12, color='black')
        )
    ]
)
# Aplicar estilo 'Clean' aos eixos
fig_irreg.update_xaxes(showgrid=False, showline=True, linewidth=1.5, linecolor='black', mirror=False)
fig_irreg.update_yaxes(showgrid=False, showline=True, linewidth=1.5, linecolor='black', mirror=False)
fig_irreg.show()


# Passo 16 - Visualização de Cruzada: Frameworks vs. Irregularidades
Nesta etapa, buscamos validar se a presença de conceitos dos frameworks (DAMA e BCBS 239) nos relatórios anuais reflete em padrões específicos de irregularidades operacionais. Utilizamos uma matriz de correlação para identificar quais pilares de governança são mais sensíveis a cada tipo de falha.

Ao cruzar o **ISM (Quantitativo)** com o **PCA/BERT (Semântico)**, observamos três comportamentos distintos:

1.  **Líderes Regulatórios (Alta Maturidade/Baixo PC1):** Bancos como **CAIXA** e **BB** mostram conformidade absoluta nos termos, mas operam em um contexto textual burocrático/institucional.
2.  **Líderes de Inovação (Alta Maturidade/Alto PC1):** Instituições como **ITAÚ** e **BRADESCO** conseguem equilibrar a densidade técnica com uma estrutura de relatório moderna.
3.  **Agilidade em Transição (Baixo ISM/Alto PC1):** Fintechs como **STONE** e **PICPAY** possuem uma linguagem altamente técnica e moderna (BERT), mas ainda não adotaram exaustivamente os termos de nicho do BCBS 239 em seus relatórios públicos.

In [ ]:
# 72. Padrões de Governança: Frameworks vs. Top Irregularidades (Estilo Passo 11)
top_irreg_list = ranking_impacto.head(20).index.tolist()
df_cross_corr = df_final_treino[['Hits_DAMA_Limpo', 'Hits_BCBS_Limpo'] + top_irreg_list]
matriz_cross = df_cross_corr.corr().loc[['Hits_DAMA_Limpo', 'Hits_BCBS_Limpo'], top_irreg_list].T
matriz_cross.columns = ['DAMA-DMBOK', 'BCBS 239']

import plotly.graph_objects as go

fig_cross = go.Figure(data=go.Heatmap(
    z=matriz_cross.values,
    x=matriz_cross.columns,
    y=matriz_cross.index,
    colorscale='RdYlGn',
    zmid=0,
    colorbar=dict(title='Correlação')
))

fig_cross.update_layout(
    height=850,
    template='plotly_white',
    title='<b>Drivers de Governança Termos dos Frameworks vs. Top Irregularidades</b><br><sup>Cruzamento entre Pilares de Maturidade e Gargalos Operacionais</sup>',
    margin=dict(t=100, b=250, l=100, r=100),
    annotations=[
        dict(
            x=-4, y=-0.35,
            showarrow=False,
            text='<b>Legenda Técnica:</b><br>O <b>Eixo X</b> representa os Frameworks Globais (DAMA para Gestão / BCBS para Risco).<br>O <b>Eixo Y</b> lista as 20 falhas operacionais com maior impacto regulatório.<br><b>Cores Verdes:</b> Correlação positiva (transparência/menção do tema). <b>Cores Vermelhas:</b> Correlação negativa.',
            xref="paper", yref="paper",
            xanchor="center", yanchor="top",
            align="center",
            font=dict(size=12, color="black")
        )
    ]
)

fig_cross.show()

In [ ]:
# 72. Padrões de Governança: Frameworks vs. Top Irregularidades (Estilo Passo 11)
top_irreg_list = ranking_impacto.head(20).index.tolist()
df_cross_corr = df_final_treino[['Hits_DAMA_Limpo', 'Hits_BCBS_Limpo'] + top_irreg_list]
matriz_cross = df_cross_corr.corr().loc[['Hits_DAMA_Limpo', 'Hits_BCBS_Limpo'], top_irreg_list].T
matriz_cross.columns = ['DAMA-DMBOK', 'BCBS 239']

import plotly.graph_objects as go

fig_cross = go.Figure(data=go.Heatmap(
    z=matriz_cross.values,
    x=matriz_cross.columns,
    y=matriz_cross.index,
    colorscale='RdYlGn',
    zmid=0,
    colorbar=dict(title='') # Título da barra de cores removido
))

fig_cross.update_layout(
    height=850,
    # template='plotly_white', # Template removido para controle manual do fundo
    plot_bgcolor='rgba(0,0,0,0)', # Remover preenchimento de fundo da área do plot
    paper_bgcolor='rgba(0,0,0,0)', # Remover preenchimento de fundo da figura inteira
    # title='<b>Drivers de Governança Termos dos Frameworks vs. Top Irregularidades</b><br><sup>Cruzamento entre Pilares de Maturidade e Gargalos Operacionais</sup>', # Título removido
    margin=dict(t=100, b=250, l=100, r=100),
    annotations=[
        dict(
            x=-4, y=-0.35,
            showarrow=False,
            text='<b>Legenda Técnica:</b><br>O <b>Eixo X</b> representa os Frameworks Globais (DAMA para Gestão / BCBS para Risco).<br>O <b>Eixo Y</b> lista as 20 falhas operacionais com maior impacto regulatório.<br><b>Cores Verdes:</b> Correlação positiva (transparência/menção do tema). <b>Cores Vermelhas:</b> Correlação negativa.',
            xref="paper", yref="paper",
            xanchor="center", yanchor="top",
            align="center",
            font=dict(size=12, color="black")
        )
    ]
)
# Aplicar estilo 'Clean' aos eixos
fig_cross.update_xaxes(showgrid=False, showline=True, linewidth=1.5, linecolor='black', mirror=False)
fig_cross.update_yaxes(showgrid=False, showline=True, linewidth=1.5, linecolor='black', mirror=False)

fig_cross.show()

Principais Insights:

Drivers de Insatisfação: Note que irregularidades como 'Crédito Consignado' e 'Registros no CCF' possuem correlações muito fortes com o índice.
Conexão Técnica: Problemas como 'Inclusão de dados incorretos no CCS' são falhas diretas de Qualidade de Dados, o que valida o uso do framework DAMA para analisar a maturidade.
Cores: As áreas em verde indicam correlações positivas (onde a presença do termo acompanha o aumento da irregularidade — o que pode indicar transparência sobre um problema existente), enquanto as áreas em vermelho indicam correlações negativas.
Você gostaria de aprofundar a análise em algum banco específico para ver como ele se comporta nestes pilares?

# Passo 17 - Modelagem: Consolidação dos Indicadores das Instituições Financeiras
Este é o 'Produto Final' do processamento. Unimos os dados de Texto (NLP/BERT), os Scores de Maturidade (ISM) e os resultados Financeiros em uma tabela única, ordenada pelos líderes de governança.

In [ ]:
# 73. Carregamento, validação e preparação dos indicadores financeiros
df_indicadores = pd.read_csv(
    '/content/00.b Lista dos principais indicadores por instituicao financeira.csv',
    sep=';',
    encoding='utf-8-sig'
)
df_indicadores.columns = [c.strip() for c in df_indicadores.columns]

colunas_numericas_indicadores = [
    'Ativos_Totais_R$Bi',
    'Clientes_Mi',
    'Valor_Mercado_R$Bi',
    'Lucro_Anual_R$Bi',
    'Indice_Resolutividade',
    'Reclamacoes_Trimestre',
    'ROE_Anual'
]
colunas_obrigatorias = ['Chave_Cruzamento', 'Instituicao', 'Conglomerado', 'Segmento_BC'] + colunas_numericas_indicadores
colunas_ausentes = [col for col in colunas_obrigatorias if col not in df_indicadores.columns]
if colunas_ausentes:
    raise ValueError(f'Colunas obrigatorias ausentes em 00.b: {colunas_ausentes}')

for coluna in colunas_numericas_indicadores:
    df_indicadores[coluna] = pd.to_numeric(
        df_indicadores[coluna].replace(['N/D', ''], np.nan),
        errors='coerce'
    )

df_indicadores['Join_Key_Ind'] = df_indicadores['Chave_Cruzamento'].astype(str).str.strip().str.upper()
if df_indicadores['Join_Key_Ind'].duplicated().any():
    chaves_duplicadas = df_indicadores.loc[
        df_indicadores['Join_Key_Ind'].duplicated(keep=False), 'Join_Key_Ind'
    ].unique().tolist()
    raise ValueError(f'Chaves_Cruzamento duplicadas em 00.b: {chaves_duplicadas}')

# Padroniza o nome oficial do conglomerado para a chave explícita de 00.b.
def simplificar_nome(nome_oficial):
    if not nome_oficial:
        return None
    nome_limpo = str(nome_oficial).upper()
    nome_limpo = nome_limpo.replace(' (CONGLOMERADO)', '').replace(' IP', '').replace(' S.A', '').strip()
    nome_limpo = nome_limpo.replace(' C.F.I', ' CFI')
    correcoes = {
        'ABC-BRASIL': 'ABC BRASIL',
        'BANCO C6': 'C6',
        'BCO DA AMAZONIA.': 'BASA',
        'BCO COOPERATIVO SICREDI': 'SICREDI',
        'BCO DAYCOVAL': 'DAYCOVAL',
        'BCO DO NORDESTE DO BRASIL.': 'BNB',
        'BB': 'BANCO DO BRASIL',
        'BRB': 'BRASÍLIA',
        'BTG PACTUAL/BANCO PAN': 'BTG PACTUAL',
        'CAIXA ECONÔMICA FEDERAL': 'CAIXA',
        'CREFISA. CFI': 'CREFISA',
        'MERCANTIL DO BRASIL': 'MERCANTIL DO BRASIL',
        'NEON PAGAMENTOS': 'NEON',
        'NU PAGAMENTOS': 'NUBANK',
        'PAGBRASIL TECNOLOGIA': 'PAGSEGURO',
        'BANCO SICOOB': 'SICOOB'
    }
    return correcoes.get(nome_limpo, nome_limpo)

# Reutiliza o mapa oficial do Passo 9 e valida cobertura antes do merge.
df_hits['Join_Key_Final'] = df_hits['Instituição_Oficial'].apply(simplificar_nome)
chaves_indicadores = set(df_indicadores['Join_Key_Ind'])
sem_indicador = df_hits.loc[
    ~df_hits['Join_Key_Final'].astype(str).str.strip().str.upper().isin(chaves_indicadores),
    'Banco_Link'
].drop_duplicates().tolist()
if sem_indicador:
    raise ValueError(f'Relatorios sem chave correspondente em 00.b: {sem_indicador}')

# 75. Merge validado: cada relatório deve encontrar no máximo uma linha financeira.
df_performance = pd.merge(
    df_hits[['Join_Key_Final', 'ISM_Bruto', 'Banco_Link', 'Arquivo']],
    df_indicadores,
    left_on='Join_Key_Final',
    right_on='Join_Key_Ind',
    how='left',
    validate='many_to_one'
)
if df_performance['Instituicao'].isna().any():
    sem_correspondencia = df_performance.loc[
        df_performance['Instituicao'].isna(), 'Banco_Link'
    ].drop_duplicates().tolist()
    raise ValueError(f'Falha inesperada no merge financeiro: {sem_correspondencia}')

# 76. Criar df_performance_unique e limpar chaves
df_performance_unique = df_performance.drop_duplicates(subset=['Banco_Link']).drop(
    columns=['Join_Key_Final', 'Join_Key_Ind', 'Chave_Cruzamento', 'Arquivo'],
    errors='ignore'
)
df_performance_unique['Banco_Link'] = df_performance_unique['Banco_Link'].astype(str).str.strip()
print(f"Indicadores financeiros associados: {df_performance_unique['Banco_Link'].nunique()} de {df_hits['Banco_Link'].nunique()} relatorios.")

# 77. Geração da Tabela Mestra Consolidada (df_mestre)
df_mestre = df_final_treino.copy()
df_mestre = pd.merge(
    df_mestre,
    df_investigacao[['Banco_Link', 'PC1', 'PC2']],
    on='Banco_Link',
    how='left',
    validate='one_to_one'
)

cols_to_add_from_performance = [
    'Instituicao', 'Conglomerado', 'Segmento_BC', 'Ativos_Totais_R$Bi',
    'Clientes_Mi', 'Valor_Mercado_R$Bi', 'Lucro_Anual_R$Bi',
    'Indice_Resolutividade', 'Reclamacoes_Trimestre', 'ROE_Anual'
]
df_mestre = pd.merge(
    df_mestre,
    df_performance_unique[['Banco_Link'] + cols_to_add_from_performance],
    on='Banco_Link',
    how='left',
    suffixes=('', '_performance_unique'),
    validate='one_to_one'
)

sem_indicadores_finais = df_mestre.loc[df_mestre['Instituicao'].isna(), 'Banco_Link'].tolist()
if sem_indicadores_finais:
    raise ValueError(f'Instituicoes sem indicadores na tabela mestra: {sem_indicadores_finais}')
df_mestre = df_mestre.sort_values(by='ISM_Bruto', ascending=False).reset_index(drop=True)

print('Geração da Tabela Mestra Consolidada (df_mestre) concluída.')
print('Amostra das top instituições por ISM_Bruto:')
display_limitado(df_mestre[['Banco_Link', 'ISM_Bruto', 'Lucro_Anual_R$Bi', 'ROE_Anual']].head())
df_mestre.to_csv('00.h Relatório Anual Integrado TCC.csv', index=False, sep=';', encoding='latin-1')


# Passo 18 - Visualização de Síntese: Maturidade vs Performance
Nesta etapa final, aplicamos a redução de dimensionalidade (PCA) sobre o dataset mestre e calculamos as correlações definitivas entre Maturidade (ISM) e Performance Financeira (Lucro/ROE).

In [ ]:
df_plot = df_mestre.dropna(subset=['Lucro_Anual_R$Bi'])

fig_final = px.scatter(
    df_plot,
    x='ISM_Bruto',
    y='Lucro_Anual_R$Bi',
    size=np.maximum(0, df_plot['ROE_Anual']), # Substituindo valores negativos por 0 para o tamanho
    text='Banco_Link',
    color='ROE_Anual',
    trendline='ols',
    size_max=45,
    color_continuous_scale='Viridis',
    title='<b>Mapa de Maturidade de Governança vs. Performance Financeira</b><br><sup>Análise Final de Correlação entre ISM e Indicadores de Rentabilidade</sup>',
    labels={
        'ISM_Bruto': 'Maturidade (ISM - Eixo X)',
        'Lucro_Anual_R$Bi': 'Lucro Anual em R$ Bilhões (Eixo Y)',
        'ROE_Anual': 'ROE Anual (%)',
        'Ativos_Totais_R$Bi': 'Tamanho do Ativo'
    },
    template='plotly_white'
)

fig_final.update_traces(textposition='top center')

fig_final.update_layout(
    height=850,
    margin=dict(t=100, b=220, l=80, r=150),
    coloraxis_colorbar=dict(
        title="Rentabilidade (ROE)",
        thicknessmode="pixels", thickness=15,
        lenmode="pixels", len=300,
        yanchor="top", y=1
    ),
    showlegend=True,
    annotations=[
        dict(
            x=0.5, y=-0.22,
            showarrow=False,
            text="<b>Legenda Técnica:</b><br>O <b>Eixo X</b> representa o ISM (Maturidade Declarada nos Relatórios).<br>O <b>Eixo Y</b> e o <b>Tamanho das Bolhas</b> representam o Lucro e a Rentabilidade (ROE), respectivamente, sendo que o ROE negativo é tratado como tamanho 0 para as bolhas.<br>A <b>Cor</b> indica a rentabilidade (ROE). A <b>Linha de Tendência</b> demonstra a correlação positiva validada estatisticamente.",
            xref='paper', yref='paper',
            xanchor='center', yanchor='top',
            align='center',
            font=dict(size=12, color='black')
        )
    ]
)
fig_final.show()

In [ ]:
# 76. Visualização Final de Síntese: Maturidade vs Performance Financeira (Estilo Passo 11)
fig_final = px.scatter(
    df_mestre.dropna(subset=['Lucro_Anual_R$Bi']),
    x='ISM_Bruto',
    y='Lucro_Anual_R$Bi',
    size='Ativos_Totais_R$Bi',
    text='Banco_Link',
    color='ROE_Anual',
    trendline='ols',
    size_max=45,
    color_continuous_scale='Viridis',
    # title='<b>Mapa de Maturidade de Governança vs. Performance Financeira</b>', # Removido título interno
    labels={
        'ISM_Bruto': 'Maturidade (ISM - Eixo X)',
        'Lucro_Anual_R$Bi': 'Lucro Anual em R$ Bilhões (Eixo Y)',
        'ROE_Anual': 'ROE Anual (%)',
        'Ativos_Totais_R$Bi': 'Tamanho do Ativo'
    }
)

fig_final.update_traces(textposition='top center')

fig_final.update_layout(
    height=850,
    margin=dict(t=100, b=220, l=80, r=150),
    # Estética Clean: Sem fundo e sem bordas
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    coloraxis_colorbar=dict(
        title="",
        thicknessmode="pixels", thickness=15,
        lenmode="pixels", len=300,
        yanchor="top", y=1
    ),
    showlegend=True,
    annotations=[
        dict(
            x=0.5, y=-0.22,
            showarrow=False,
            text="<b>Legenda Técnica:</b><br>O <b>Eixo X</b> representa o ISM (Maturidade Declarada nos Relatórios).<br>O <b>Eixo Y</b> e o <b>Tamanho das Bolhas</b> representam o Lucro e Ativos Totais, respectivamente.<br>A <b>Cor</b> indica a rentabilidade (ROE). A <b>Linha de Tendência</b> demonstra a correlação positiva validada estatisticamente.",
            xref='paper', yref='paper',
            xanchor='center', yanchor='top',
            align='center',
            font=dict(size=12, color='black')
        )
    ]
)

# Eixos: Linhas sólidas pretas com largura de 1,5 pt e sem linhas de grade
fig_final.update_xaxes(
    showgrid=False,
    showline=True,
    linewidth=1.5,
    linecolor='black',
    mirror=False
)

fig_final.update_yaxes(
    showgrid=False,
    showline=True,
    linewidth=1.5,
    linecolor='black',
    mirror=False
)

fig_final.show()


In [ ]:
print('--- Matriz de Correlação Final (Significância Acadêmica) ---')
display_limitado(df_mestre[['ISM_Bruto', 'Lucro_Anual_R$Bi', 'Ativos_Totais_R$Bi', 'ROE_Anual']].corr())

### Validação Científica: Teste de Significância
Para que a análise seja aceita academicamente, precisamos provar que a correlação não é casual. Utilizamos o **Coeficiente de Pearson (r)** para medir a força da relação e o **P-Valor** para medir a probabilidade de erro.

In [ ]:
# 78. Teste de significância estatística (Maturidade vs. Lucro)
dados_para_teste = df_mestre[['ISM_Bruto', 'Lucro_Anual_R$Bi']].copy()
dados_para_teste['ISM_Bruto'] = pd.to_numeric(dados_para_teste['ISM_Bruto'], errors='coerce')
dados_para_teste['Lucro_Anual_R$Bi'] = pd.to_numeric(
    dados_para_teste['Lucro_Anual_R$Bi'], errors='coerce'
)
dados_para_teste = dados_para_teste.dropna(subset=['ISM_Bruto', 'Lucro_Anual_R$Bi'])
dados_para_teste = dados_para_teste[dados_para_teste['Lucro_Anual_R$Bi'] != 0]

if len(dados_para_teste) >= 2 and dados_para_teste['ISM_Bruto'].nunique() > 1 and dados_para_teste['Lucro_Anual_R$Bi'].nunique() > 1:
    corr_coef, p_valor = pearsonr(
        dados_para_teste['ISM_Bruto'],
        dados_para_teste['Lucro_Anual_R$Bi']
    )
    print('--- Resultados Estatísticos para o TCC ---')
    print(f'Instituições analisadas: {len(dados_para_teste)}')
    print(f'Coeficiente de Correlação (r): {corr_coef:.4f}')
    print(f'P-Valor (p): {p_valor:.4f}')
    if p_valor < 0.05:
        print('\n✅ CONCLUSÃO: Relação estatisticamente significante.')
    else:
        print('\n⚠️ CONCLUSÃO: Relação existente, mas com significância marginal (p > 0.05).')
else:
    print('⚠️ Dados numéricos válidos e com variação insuficientes para realizar o teste de Pearson.')
